# PubMed Oncology Gemma 4 12B SFT Fine-Tuning with Unsloth (4-bit QLoRA)

**Base Model:** Gemma 4 12B Instruct (`unsloth/gemma-4-12b-it`)

**Dataset:** PubMed oncology multi-turn and tool-calling SFT conversations across 11 cancer types

**Training Hardware:** NVIDIA DGX Spark (128 GB unified memory)

**Chat Template:** Tokenizer-native Gemma 4 template, including native tool-call and tool-response serialization

**Architecture:** Phase 1 SFT. Phase 2 DPO refines response preferences using the paired PubMed oncology dataset.

## 1. Setup — Configuration, Environment, GPU Check

Installs missing packages, verifies GPU, sets all paths and hyperparameters. Safe to re-run.

In [1]:
import os
from pathlib import Path

os.environ["UNSLOTH_ENABLE_FLEX_ATTENTION"] = "0"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "garbage_collection_threshold:0.5,max_split_size_mb:256"

import torch
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU available. Run this in the Unsloth notebook container.")

_MEMORY_FRACTION = 0.55
try:
    torch.cuda.set_per_process_memory_fraction(_MEMORY_FRACTION, 0)
except RuntimeError as error:
    print(f"CUDA memory fraction was not set: {error}")

import unsloth
import transformers

if os.path.exists("/workspace/training/pubmed"):
    PROJECT_ROOT = Path("/workspace/training/pubmed")
elif os.path.exists("/workspace/pubmed"):
    PROJECT_ROOT = Path("/workspace/pubmed")
else:
    PROJECT_ROOT = Path("/home/spark/projects/training/pubmed")

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_ROOT = PROJECT_ROOT / "output" / "v3"
BASE_LLM = "unsloth/gemma-4-12b-it"
MODEL_NAME_BASE = "pubmed_oncologist_v3_gemma4_12b_sft"
INPUT_DATA_FILE = DATA_DIR / "training-data" / "pubmed_oncologist_v2_tool_sft_messages.jsonl"
OUTPUT_BASE_DIR = OUTPUT_ROOT / MODEL_NAME_BASE
TRAIN_DIR = OUTPUT_BASE_DIR / "train"
LORA_OUTPUT_DIR = OUTPUT_BASE_DIR / "lora_adapters"

MAX_SEQ_LENGTH = 16384
BATCH_SIZE = 2
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4
TARGET_EPOCHS = 1
LORA_R = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]
TEST_PROMPT = "A 58-year-old woman with BRCA1-mutated high-grade serous ovarian cancer has progressed after platinum-based chemotherapy and a PARP inhibitor. What are the next treatment options?"

for path, label in [(INPUT_DATA_FILE, "Training data"), (PROJECT_ROOT, "Project root")]:
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Base model: {BASE_LLM}")
print(f"Input data: {INPUT_DATA_FILE}")
print(f"Output: {OUTPUT_BASE_DIR}")
print(f"CUDA memory fraction: {_MEMORY_FRACTION}")

ENVIRONMENT SETUP
  torch 2.10.0a0+b558c986e8.nv25.11 — CUDA 13.0 — GPU: NVIDIA GB10
  UNSLOTH_ENABLE_FLEX_ATTENTION = 0
  causal_conv1d: OK (CUDA extension loaded)
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
  transformers 5.10.0.dev0

  unsloth                   2026.5.7             [OK]
  transformers              5.10.0.dev0          [OK]
  trl                       0.24.0               [OK]
  causal_conv1d             0.0.local            [OK]

CONFIGURATION
  Environment:  Docker (Unsloth container)
  PROJECT_ROOT: /workspace/training/pubmed
  Base model:   unsloth/gemma-4-12b-it
  Model name:   pubmed_oncologist_v3_gemma4_12b_sft
  Input data:   /workspace/training/pubmed/data/training-data/pubmed_oncologist_v2_tool_sft_messages.jsonl
  LoRA output:  /workspace/training/pubmed/output/v3/pubmed_oncologist_v3_gemma4_12b_sft/lora_adapters
  LoRA:         r=32, alpha=32, targets=7 modules
 

## 2. Load Dataset

Load the combined multi-turn ShareGPT JSONL from datagen.

- 33,349 conversations across 11 cancer types
- Data types: QA (with thinking), continuation, treatment reasoning, beyond-evidence, self-correction
- Standard ShareGPT format: `[system, human, gpt, human, gpt, ...]`
- System prompts are extracted from the JSONL at load time (stays in sync with datagen)

In [2]:
import json
from collections import defaultdict
from datasets import Dataset as HFDataset

SFT_MAX_EXAMPLES = 9000
raw_rows = []
cancer_type_counts = defaultdict(int)
source_counts = defaultdict(int)
system_prompts_by_cancer = {}

role_map = {"system": "system", "human": "user", "gpt": "assistant"}
with INPUT_DATA_FILE.open() as handle:
    for line in handle:
        row = json.loads(line)
        cancer = row.get("cancer_type", "unknown")
        source = row.get("source", "unknown")
        cancer_type_counts[cancer] += 1
        source_counts[source] += 1

        if "messages" in row:
            messages = row["messages"]
        elif "conversations" in row:
            messages = [
                {"role": role_map[turn["from"]], "content": turn["value"]}
                for turn in row["conversations"]
            ]
        else:
            raise ValueError("Row missing both messages and conversations")

        if messages and messages[0].get("role") == "system":
            system_prompt = messages[0].get("content", "")
            if system_prompt and cancer not in system_prompts_by_cancer:
                system_prompts_by_cancer[cancer] = system_prompt
        raw_rows.append({"messages": messages})

if SFT_MAX_EXAMPLES is not None and len(raw_rows) > SFT_MAX_EXAMPLES:
    raw_rows = raw_rows[:SFT_MAX_EXAMPLES]

dataset = HFDataset.from_list(raw_rows)
print(f"Loaded {len(dataset):,} conversations")
print(f"Cancer types: {len(cancer_type_counts)}")
print(f"System prompts: {len(system_prompts_by_cancer)}")
for source, count in sorted(source_counts.items(), key=lambda item: -item[1]):
    print(f"  {source}: {count}")

LOADING TOOL-CALLING SFT DATA
  File: /workspace/training/pubmed/data/training-data/pubmed_oncologist_v2_tool_sft_messages.jsonl
  Capping raw dataset size from 19,293 to SFT_MAX_EXAMPLES=9,000

Total dataset: 9000 conversations
Cancer types: 11
Sources: 2
Unique system prompts: 11
Columns: ['messages']

Per-cancer breakdown (entire file):
  pubmed_brain_tumour             1788 conversations
  pubmed_bone_cancer              1788 conversations
  pubmed_prostate_cancer          1787 conversations
  pubmed_lung_cancer              1787 conversations
  pubmed_colon_cancer             1787 conversations
  pubmed_ovarian_cancer           1787 conversations
  pubmed_skin_cancer              1787 conversations
  pubmed_breast_cancer            1787 conversations
  pubmed_kidney_cancer            1721 conversations
  pubmed_gastric_cancer           1694 conversations
  cancerguide_treatment_reasoning  1580 conversations

Per-source breakdown:
  tool_calling_augmentation       9647 rows
  direc

## 3. Validate & Summarize Dataset

Verify data quality: turn structure, non-empty responses, thinking tag presence.

In [3]:
from collections import Counter

bad_examples = []
empty_final_responses = []
tool_call_examples = 0
for index, example in enumerate(dataset):
    messages = example["messages"]
    if len(messages) < 3:
        bad_examples.append((index, "expected at least three messages"))
        continue
    if messages[0].get("role") != "system" or messages[1].get("role") != "user":
        bad_examples.append((index, "expected system then user"))
        continue
    final_message = messages[-1]
    if final_message.get("role") != "assistant" or not str(final_message.get("content", "")).strip():
        empty_final_responses.append(index)
    if any(message.get("role") == "assistant" and message.get("tool_calls") for message in messages):
        tool_call_examples += 1

if bad_examples:
    raise ValueError(f"Invalid message structure in {len(bad_examples)} rows; first: {bad_examples[:5]}")
if empty_final_responses:
    dataset = dataset.select([index for index in range(len(dataset)) if index not in set(empty_final_responses)])

role_sequences = Counter(tuple(message.get("role") for message in row["messages"]) for row in dataset)
print(f"Validated {len(dataset):,} conversations")
print(f"Tool-call conversations: {tool_call_examples:,}")
print(f"Top role sequences: {role_sequences.most_common(5)}")

DATA QUALITY CHECK
  Total examples: 9000
  Bad structure: 0
  Empty final responses: 0
  With explicit tool_calls: 4552 (50.6%)
  Without explicit tool_calls: 4448 (49.4%)

Top role sequences (up to 5):
  ('system', 'user', 'assistant', 'tool', 'assistant') -> 4552
  ('system', 'user', 'assistant') -> 4448

CANCER TYPE DISTRIBUTION:
  pubmed_brain_tumour              1788  ########
  pubmed_bone_cancer               1788  ########
  pubmed_prostate_cancer           1787  ########
  pubmed_lung_cancer               1787  ########
  pubmed_colon_cancer              1787  ########
  pubmed_ovarian_cancer            1787  ########
  pubmed_skin_cancer               1787  ########
  pubmed_breast_cancer             1787  ########
  pubmed_kidney_cancer             1721  ########
  pubmed_gastric_cancer            1694  ########
  cancerguide_treatment_reasoning  1580  #######
  TOTAL                           19293

Dataset validated and ready for training


## 4. Load Model & Tokenizer (4-bit)

In [4]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_LLM,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
if hasattr(tokenizer, "tokenizer"):
    tokenizer = tokenizer.tokenizer
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id

print(f"Model loaded: {BASE_LLM}")
print(f"Precision: 4-bit QLoRA")
print(f"Max sequence length: {MAX_SEQ_LENGTH}")

==((====))==  Unsloth 2026.5.7: Fast Gemma3 patching. Transformers: 5.10.0.dev0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.689 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260302. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/808 [00:00<?, ?it/s]

  Attention override: flash_attention_2 -> flash_attention_2
  pad_token = '<pad>' (id=0)
Model loaded: unsloth/gemma-4-12b-it
  Precision: 4-bit QLoRA (pre-quantized NF4)
  Max sequence length: 16384
  Vocab size: 262145
  GPU allocated: 16.6 GB


## 5. Format Dataset for Chat Template

Map ShareGPT roles to Gemma 3 chat template roles, apply the tokenizer's native chat template, then manual sequence packing for 100% token utilization.

**Gemma role mapping:** `system` → `system`, `human` → `user`, `gpt` → `model`

**Note:** The training data contains `<think>` reasoning blocks in the GPT responses. These are treated as regular content — MedGemma will learn to produce structured reasoning in this format during fine-tuning.


In [5]:
from datasets import Dataset as HFDataset

formatted_texts = tokenizer.apply_chat_template(
    list(dataset["messages"]),
    tokenize=False,
    enable_thinking=False,
)
dataset = HFDataset.from_dict({"text": formatted_texts})
dataset = dataset.filter(lambda row: bool(row["text"].strip()))
dataset = dataset.shuffle(seed=42)

print(f"Formatted {len(dataset):,} conversations with the native Gemma 4 template")
print(dataset[0]["text"][:500])

Formatting and Mapping roles:   0%|          | 0/9000 [00:00<?, ?it/s]

Applying chat template...


Tokenizing & packing:   0%|          | 0/9000 [00:00<?, ?it/s]

Dataset packed: 9000 conversations -> 1715 chunks of 16384 tokens
  Total tokens: 28,105,181  |  Wasted (tail): 6,621 (0.0%)
  Token utilization: ~100% (no padding)

--- Sample packed text (first 500 chars) ---
 For biomedical literature, oncology, clinical evidence, trial, biomarker, mechanism, treatment, guideline, or PubMed-style questions, call `oncology_guidelines_db` before answering.
- When you call `oncology_guidelines_db`, do not answer in the same assistant turn. Wait for the tool result, then synthesize the answer from the returned evidence.
- Do not invent PMIDs, trial names, statistics, guideline statements, or article details. If the tool result is incomplete, say what remains uncertain.



## 6. Add LoRA Adapters

In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    max_seq_length=MAX_SEQ_LENGTH,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"LoRA adapters added (r={LORA_R}, alpha={LORA_ALPHA})")
print(f"  Trainable: {trainable:,} / {total:,} params ({100*trainable/total:.2f}%)")
print(f"  Target modules: {LORA_TARGET_MODULES}")

/usr/local/lib/python3.12/dist-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)


WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.


INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.          


fatal: detected dubious ownership in repository at '/workspace/training/pubmed'
To add an exception for this directory, call:

	git config --global --add safe.directory /workspace/training/pubmed


INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 7.0.0
Transformers : 5.10.0.dev0
Torch        : 2.10.0a0+b558c986e8.nv25.11
Triton       : 3.4.0+gitc5d671f9


LoRA adapters added (r=32, alpha=32)
  Trainable: 227,033,088 / 14,610,262,784 params (1.55%)
  Target modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']


## 7. Trainer Setup

In [7]:
import gc
from transformers import TrainerCallback
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_length=MAX_SEQ_LENGTH,
        packing=True,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=5,
        num_train_epochs=TARGET_EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        gradient_checkpointing=True,
        dataloader_pin_memory=False,
        output_dir=str(TRAIN_DIR),
        save_strategy="steps",
        save_steps=100,
        save_total_limit=3,
        report_to="none",
        dataset_num_proc=1,
    ),
)

class CudaCacheClearCallback(TrainerCallback):
    def on_step_begin(self, args, state, control, **kwargs):
        torch.cuda.empty_cache()
        gc.collect()

    def on_step_end(self, args, state, control, **kwargs):
        torch.cuda.empty_cache()
        gc.collect()

trainer.add_callback(CudaCacheClearCallback())
print(f"Trainer configured for {len(dataset):,} conversations")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")

Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/1715 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/multiprocess/popen_fork.py:66: DeprecationWarning: This process (pid=579) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()


Trainer configured (PubMed Oncology Gemma 4 12B 4-bit QLoRA — pre-packed)
  Effective batch size: 2 x 4 = 8
  Epochs: 1  |  Steps: ~215
  LR: 0.0002
  Loss Masking: Active via CustomCompletionCollator (template: '<start_of_turn>model\n')
  Packing: manual (pre-packed, each example = 16384 tokens, zero padding)
  Precision: bf16
  Dataset: 1715 packed chunks


## 8. Train

In [8]:
from transformers.trainer_utils import get_last_checkpoint

last_checkpoint = get_last_checkpoint(str(TRAIN_DIR))
if last_checkpoint:
    print(f"Resuming from checkpoint: {last_checkpoint}")
    result = trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    result = trainer.train()

print(f"Training complete: loss={result.training_loss:.4f}, steps={result.global_step}")

No previous checkpoint found — starting fresh.


[transformers] ==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,715 | Num Epochs = 1 | Total steps = 215
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 227,033,088 of 27,236,035,328 (0.83% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,7.318943
2,7.358215
3,7.132417
4,6.549130
5,5.639441
6,5.230318
7,4.793527
8,4.543701
9,4.092892
10,3.778145


[transformers] Unsloth: Restored added_tokens_decoder metadata in /workspace/training/pubmed/output/v3/pubmed_oncologist_v3_gemma4_12b_sft/train/checkpoint-100/tokenizer_config.json.
[transformers] Unsloth: Preserved sentencepiece asset `tokenizer.model` in /workspace/training/pubmed/output/v3/pubmed_oncologist_v3_gemma4_12b_sft/train/checkpoint-100.
[transformers] Unsloth: Restored added_tokens_decoder metadata in /workspace/training/pubmed/output/v3/pubmed_oncologist_v3_gemma4_12b_sft/train/checkpoint-200/tokenizer_config.json.
[transformers] Unsloth: Preserved sentencepiece asset `tokenizer.model` in /workspace/training/pubmed/output/v3/pubmed_oncologist_v3_gemma4_12b_sft/train/checkpoint-200.
[transformers] Unsloth: Restored added_tokens_decoder metadata in /workspace/training/pubmed/output/v3/pubmed_oncologist_v3_gemma4_12b_sft/train/checkpoint-215/tokenizer_config.json.
[transformers] Unsloth: Preserved sentencepiece asset `tokenizer.model` in /workspace/training/pubmed/output/v3


✓ Training complete!
  Final loss:     1.0030
  Total steps:    215
  Training time:  1502.4 minutes


## 9. Save LoRA Adapters

Save the trained LoRA adapters and system prompts. The DPO notebook (Phase 2) expects the LoRA at this path.

In [9]:
import json
from pathlib import Path

LORA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(LORA_OUTPUT_DIR))
tokenizer.save_pretrained(str(LORA_OUTPUT_DIR))

prompts_path = LORA_OUTPUT_DIR / "oncologist_system_prompts.json"
prompts_path.write_text(json.dumps(system_prompts_by_cancer, indent=2) + "\
", encoding="utf-8")
print(f"Saved Gemma 4 SFT LoRA to {LORA_OUTPUT_DIR}")

Saving LoRA adapters to /workspace/training/pubmed/output/v3/pubmed_oncologist_v3_gemma4_12b_sft/lora_adapters...


[transformers] Unsloth: Restored added_tokens_decoder metadata in /workspace/training/pubmed/output/v3/pubmed_oncologist_v3_gemma4_12b_sft/lora_adapters/tokenizer_config.json.
[transformers] Unsloth: Preserved sentencepiece asset `tokenizer.model` in /workspace/training/pubmed/output/v3/pubmed_oncologist_v3_gemma4_12b_sft/lora_adapters.



LoRA adapters saved!
  Adapters:       /workspace/training/pubmed/output/v3/pubmed_oncologist_v3_gemma4_12b_sft/lora_adapters
  System prompts: /workspace/training/pubmed/output/v3/pubmed_oncologist_v3_gemma4_12b_sft/lora_adapters/oncologist_system_prompts.json (11 cancer types)
  DPO notebook expects LoRA at: {PROJECT_ROOT}/output/v3/{MODEL_NAME_BASE}/lora_adapters
  README.md                                     0.0 MB
  adapter_config.json                           0.0 MB
  adapter_model.safetensors                   866.2 MB
  chat_template.jinja                           0.0 MB
  oncologist_system_prompts.json                0.0 MB
  tokenizer.json                               31.8 MB
  tokenizer.model                               4.5 MB
  tokenizer_config.json                         1.1 MB


## 10. Test Inference

Smoke test with oncology questions across different cancer types. The training data includes `<think>` reasoning blocks, so the model may produce them in responses.


In [10]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_LLM,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
if hasattr(tokenizer, "tokenizer"):
    tokenizer = tokenizer.tokenizer
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id

print(f"Model loaded: {BASE_LLM}")
print(f"Precision: 4-bit QLoRA")
print(f"Max sequence length: {MAX_SEQ_LENGTH}")

[transformers] Both `max_new_tokens` (=2048) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFERENCE TEST — 3 CANCER TYPES

  CANCER TYPE: CANCERGUIDE_TREATMENT_REASONING
  Q: A 58-year-old woman with BRCA1-mutated high-grade serous ovarian cancer has progressed after platinum-based chemotherapy and a PARP inhibitor. What are the next treatment options?
  A: <think>
Okay, let me approach this systematically as a clinical oncologist. The patient is a 58-year-old woman with BRCA1-mutated high-grade serous ovarian cancer (HGSOC) who has progressed after standard first-line therapy: platinum-based chemotherapy and PARP inhibitor maintenance. 

First, I need to confirm the key clinical details: 
- BRCA1 mutation confirmed (critical for PARP inhibitor response)
- HGSOC histology (most common ovarian cancer subtype)
- Progression after platinum + PARP inhibitor (likely maintenance phase)
- Age 58 (generally fit for further treatment)

The question asks for next treatment options, so I should structure my response around evidence-based pathways while acknowledging limitations. 

Hmm

[transformers] Both `max_new_tokens` (=2048) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



  CANCER TYPE: PUBMED_PROSTATE_CANCER
  Q: A 58-year-old woman with BRCA1-mutated high-grade serous ovarian cancer has progressed after platinum-based chemotherapy and a PARP inhibitor. What are the next treatment options?
  A: <think>
Okay, let me approach this systematically as a clinical oncologist. The user presents a complex case of a 58-year-old woman with advanced ovarian cancer who has progressed after platinum-based chemotherapy and PARP inhibitor treatment. 

First, I need to verify the key details: she has BRCA1-mutated high-grade serous ovarian cancer (HGSOC), which is crucial because BRCA status dictates treatment options. She's progressed after two lines of therapy - platinum chemo and a PARP inhibitor. The question asks for next treatment options.

Hmm, this is a classic scenario where we must consider the molecular profile first. BRCA1 mutation means she should have responded to platinum initially (which is standard for HGSOC), but now she's progressed. The PARP inhibi

[transformers] Both `max_new_tokens` (=2048) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



  CANCER TYPE: PUBMED_KIDNEY_CANCER
  Q: A 58-year-old woman with BRCA1-mutated high-grade serous ovarian cancer has progressed after platinum-based chemotherapy and a PARP inhibitor. What are the next treatment options?
  A: <think>
Okay, let me approach this systematically as a clinical oncologist would. The user presents a complex case: a 58-year-old woman with BRCA1-mutated high-grade serous ovarian cancer (HGSOC) who has progressed after platinum-based chemotherapy and a PARP inhibitor. 

First, I need to unpack the clinical scenario. The patient has BRCA1-mutated HGSOC - that's crucial because it's a homologous recombination deficiency (HRD) tumor. She's already received platinum-based chemo and a PARP inhibitor (likely olaparib or niraparib based on FDA approvals), which is standard first-line maintenance after response to platinum. Now she's progressed, so we're in the relapsed/refractory setting.

Hmm... the key question is about next treatment options. I should start by reca

## 11. Verify Adapter (Cold Reload)

Loads adapters from disk in a fresh model to confirm portability. Also verifies the DPO notebook can load these adapters.

In [11]:
import gc, torch, json
from pathlib import Path

del model, tokenizer, trainer, dataset
gc.collect()
torch.cuda.empty_cache()

print("Cleared training model from memory")
print(f"  Loading adapter from: {LORA_OUTPUT_DIR}")

from unsloth import FastLanguageModel

model2, tokenizer2 = FastLanguageModel.from_pretrained(
    model_name=LORA_OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
model2.config._attn_implementation = "flash_attention_2"
FastLanguageModel.for_inference(model2)

with open(f"{LORA_OUTPUT_DIR}/oncologist_system_prompts.json") as f:
    reloaded_prompts = json.load(f)

test_cancer = list(reloaded_prompts.keys())[0]
test_system = reloaded_prompts[test_cancer]

messages = [
    {"role": "system", "content": test_system},
    {"role": "user", "content": TEST_PROMPT},
]

# Gemma 3 chat template
text = tokenizer2.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer2(text=text, return_tensors=\"pt\").to(model2.device)

outputs = model2.generate(
    **inputs,
    max_new_tokens=2048,
    temperature=0.7,
    top_p=0.8,
    do_sample=True,
)

response = tokenizer2.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

print(f"\nADAPTER RELOAD TEST (cancer type: {test_cancer}):")
print(f"  Q: {TEST_PROMPT}")
print(f"  A: {response[:500]}")
print(f"\nAdapter loads cleanly from disk.")
print(f"DPO notebook (Phase 2) can now load this LoRA from: {LORA_OUTPUT_DIR}")

print(f"\nAdapter contents:")
for p in sorted(Path(LORA_OUTPUT_DIR).iterdir()):
    size_mb = p.stat().st_size / 1024 / 1024
    print(f"  {p.name:40s} {size_mb:>8.1f} MB")

del model2, tokenizer2, inputs, outputs
gc.collect()
torch.cuda.empty_cache()

Cleared training model from memory
  Loading adapter from: /workspace/training/pubmed/output/v3/pubmed_oncologist_v3_gemma4_12b_sft/lora_adapters
==((====))==  Unsloth 2026.5.7: Fast Gemma3 patching. Transformers: 5.10.0.dev0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.689 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260302. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=579) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()


Loading weights:   0%|          | 0/808 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=2048) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ADAPTER RELOAD TEST (cancer type: cancerguide_treatment_reasoning):
  Q: A 58-year-old woman with BRCA1-mutated high-grade serous ovarian cancer has progressed after platinum-based chemotherapy and a PARP inhibitor. What are the next treatment options?
  A: <think>
Okay, let me approach this systematically as a clinical oncologist. The patient is a 58-year-old woman with BRCA1-mutated high-grade serous ovarian cancer (HGSOC) who has progressed after platinum-based chemotherapy and a PARP inhibitor. This is a classic case where we need to consider maintenance therapy options beyond initial treatment.

First, I need to recall the standard treatment sequence for BRCA-mutated HGSOC. Typically, first-line treatment is platinum-based chemo (like carbopla

Adapter loads cleanly from disk.
DPO notebook (Phase 2) can now load this LoRA from: /workspace/training/pubmed/output/v3/pubmed_oncologist_v3_gemma4_12b_sft/lora_adapters

Adapter contents:
  README.md                                     